In [1]:
from pathlib import Path

import numpy as np
import open3d as o3d
import pymeshlab
import matplotlib.pyplot as plt

from plant_shape_analysis.dataloaders.trackplant3D import LeafSequencesDataset
from plant_shape_analysis.data_preprocessing.leaf_meshing import mesh_leaf

from plant_shape_analysis.reconstruction import classical_methods

%load_ext autoreload
%autoreload 2

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Mesh example single leaf

In [ ]:
dataset_path = Path("../data/TrackPlant3D")
leaf_dataset = LeafSequencesDataset(
    dataset_path, 
    min_timepoints=1, # we get all leaves
)

In [ ]:
maize_leaves = leaf_dataset.get_timeseries_by_crop('maize')
len(maize_leaves)

In [ ]:
seq_id = 4
timepoint = 0
leaf = maize_leaves[seq_id]['timepoints'][timepoint]
leaf

In [ ]:
pcd, mesh = mesh_leaf(leaf['points'], method="ball_pivoting", backend="pymeshlab")
mesh.get_surface_area()

In [ ]:
pcd, mesh = mesh_leaf(leaf['points'], method="ball_pivoting", backend="open3d")
mesh.get_surface_area()

In [ ]:
pcd, mesh = mesh_leaf(leaf['points'], method="poisson", backend="pymeshlab")
mesh.get_surface_area()

In [ ]:
pcd, mesh = mesh_leaf(leaf['points'], method="poisson", backend="open3d")
mesh.get_surface_area()

In [ ]:
o3d.visualization.draw_geometries(
[mesh],
point_show_normal=True,
window_name="Leaf Point Cloud and normals",
)

## Extract all dense leaves

In [ ]:
dataset_path = Path("../data/TrackPlant3D")
leaf_dataset = LeafSequencesDataset(
    dataset_path, 
    min_timepoints=1, # we get all leaves
)

In [ ]:
# for seq in leaf_dataset:
#     for scan in seq['timepoints']:
#         if scan['dense_points'] is not None:
#             # save leaf
#             crop = ''.join([c for c in seq['sequence_name'].split('_')[0] if c.isalpha()])
#             file_name = scan['file_path'].name.replace(".txt", f"_leaf{scan['leaf_id']}_dense.ply")
#             pcd = o3d.geometry.PointCloud()
#             pcd.points = o3d.utility.Vector3dVector(scan['dense_points'])
#             save_dir = Path(f"../data/TrackPlant3D/dense_leaves/{crop}")
#             save_dir.mkdir(parents=True, exist_ok=True)
#             o3d.io.write_point_cloud(save_dir / file_name, pcd)

## Trimmed Poission

In [2]:
# Load and visualize a sample leaf point cloud
#sample_leaf_path = Path("../data/TrackPlant3D/dense_leaves/maize/19_maize_control_plant3_D00_leaf1_dense.ply")
sample_leaf_path = Path("../data/TrackPlant3D/examples/single_leafs/maize/1_maize_control_plant1_D00_leaf1_dense.ply")
#sample_leaf_path = Path("../data/TrackPlant3D/examples/single_leafs/maize/1_maize_control_plant1_D00_leaf1_dense_downsampled-voxel02.ply")
leaf_pcd = o3d.io.read_point_cloud(str(sample_leaf_path))

leaf_pcd.estimate_normals()
leaf_pcd.orient_normals_to_align_with_direction()
leaf_pcd.orient_normals_consistent_tangent_plane(k=50)
leaf_np = np.asarray(leaf_pcd.points)
point_cloud = np.hstack([leaf_np, np.asarray(leaf_pcd.normals)])
print(f"Loaded leaf point cloud with {leaf_np.shape[0]} points.")
o3d.visualization.draw_geometries([leaf_pcd])

Loaded leaf point cloud with 155504 points.


In [ ]:
pcd, mesh = mesh_leaf(leaf_np, method="poisson", backend="pymeshlab")

Why does it look good (grey) and with normals?

In [ ]:
ms = pymeshlab.MeshSet()
mesh = pymeshlab.Mesh(vertex_matrix=leaf_np)
ms.add_mesh(mesh, "leaf")
ms.compute_normal_for_point_clouds(k=20, smoothiter=10)
ms.generate_surface_reconstruction_screened_poisson(depth=8)
mesh = ms.current_mesh()

vertices = np.asarray(mesh.vertex_matrix())
triangles = np.asarray(mesh.face_matrix())
o3d_mesh = o3d.geometry.TriangleMesh()
o3d_mesh.vertices = o3d.utility.Vector3dVector(vertices)
o3d_mesh.triangles = o3d.utility.Vector3iVector(triangles)
#o3d_mesh.compute_vertex_normals() # this makes faces look good
o3d.visualization.draw_geometries([o3d_mesh])

Try poisson with open3d, and try to filter out points based on density

In [ ]:
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(leaf_np)

# Estimate normals
pcd.estimate_normals()
pcd.orient_normals_to_align_with_direction()
pcd.orient_normals_consistent_tangent_plane(k=30)  # This is very important!!

# Create mesh using Poisson Surface Reconstruction
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd, depth=8, width=0, scale=1.1, linear_fit=False
)
mesh.vertex_colors = mesh.vertex_normals

In [ ]:
densities = np.asarray(densities)
density_colors = plt.cm.plasma(densities / np.max(densities))[:, :3]  # Use colormap
mesh.vertex_colors = o3d.utility.Vector3dVector(density_colors)
o3d.visualization.draw_geometries([mesh])

In [ ]:
def analyze_densities(densities):
    """Analyze density distribution and help choose threshold"""
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Histogram of densities
    axes[0,0].hist(densities, bins=100, alpha=0.7, edgecolor='black')
    axes[0,0].set_xlabel('Density Value')
    axes[0,0].set_ylabel('Frequency')
    axes[0,0].set_title('Distribution of Density Values')
    axes[0,0].grid(True, alpha=0.3)
    
    # Add percentile lines
    percentiles = [5, 10, 20, 50, 80, 90, 95]
    for p in percentiles:
        val = np.percentile(densities, p)
        axes[0,0].axvline(val, color='red', linestyle='--', alpha=0.6)
        axes[0,0].text(val, axes[0,0].get_ylim()[1]*0.9, f'{p}th', 
                      rotation=90, ha='right', va='top')
    
    # 2. Log-scale histogram (often better for density distributions)
    axes[0,1].hist(densities, bins=100, alpha=0.7, edgecolor='black')
    axes[0,1].set_xlabel('Density Value')
    axes[0,1].set_ylabel('Frequency (log scale)')
    axes[0,1].set_title('Distribution of Density Values (Log Scale)')
    axes[0,1].set_yscale('log')
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. Cumulative distribution
    sorted_densities = np.sort(densities)
    cumulative = np.arange(1, len(sorted_densities) + 1) / len(sorted_densities)
    axes[1,0].plot(sorted_densities, cumulative)
    axes[1,0].set_xlabel('Density Value')
    axes[1,0].set_ylabel('Cumulative Probability')
    axes[1,0].set_title('Cumulative Distribution Function')
    axes[1,0].grid(True, alpha=0.3)
    
    # Add threshold suggestions
    for p in [10, 20, 30]:
        val = np.percentile(densities, p)
        axes[1,0].axvline(val, color='red', linestyle='--', alpha=0.6)
        axes[1,0].text(val, p/100, f'{p}th percentile\n{val:.6f}', 
                      ha='left', va='bottom', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # 4. Box plot
    axes[1,1].boxplot(densities, vert=True)
    axes[1,1].set_ylabel('Density Value')
    axes[1,1].set_title('Box Plot of Densities')
    axes[1,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("Density Statistics:")
    print(f"Min: {np.min(densities):.8f}")
    print(f"Max: {np.max(densities):.8f}")
    print(f"Mean: {np.mean(densities):.8f}")
    print(f"Std: {np.std(densities):.8f}")
    print(f"Median: {np.median(densities):.8f}")
    print("\nPercentile values:")
    for p in [1, 5, 10, 20, 30, 50]:
        print(f"{p:2d}th percentile: {np.percentile(densities, p):.8f}")

# Use the function
analyze_densities(densities)

In [ ]:
def test_thresholds(mesh, densities, thresholds=None):
    """Test different density thresholds and visualize results"""
    
    if thresholds is None:
        # Default thresholds based on percentiles
        thresholds = [
            np.percentile(densities, p) for p in [5, 10, 15, 20, 25, 30]
        ]
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    original_vertices = len(np.asarray(mesh.vertices))
    
    for i, threshold in enumerate(thresholds):
        # Calculate how many vertices would be removed
        vertices_to_remove = densities < threshold
        remaining_vertices = np.sum(~vertices_to_remove)
        percentage_kept = (remaining_vertices / original_vertices) * 100
        
        # Plot histogram
        axes[i].hist(densities, bins=100, alpha=0.5, color='blue', label='All densities')
        axes[i].axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold:.6f}')
        
        # Fill area to be removed
        x_fill = np.linspace(np.min(densities), threshold, 100)
        y_fill = np.histogram(densities, bins=100)[0]
        bin_edges = np.histogram(densities, bins=100)[1]
        
        # Find which bins are below threshold
        below_threshold = bin_edges[:-1] <= threshold
        if np.any(below_threshold):
            axes[i].bar(bin_edges[:-1][below_threshold], y_fill[below_threshold], 
                       width=np.diff(bin_edges)[0], alpha=0.3, color='red', label='Removed')
        
        axes[i].set_title(f'Threshold: {threshold:.6f}\nKeeping {percentage_kept:.1f}% of vertices')
        axes[i].set_xlabel('Density')
        axes[i].set_ylabel('Frequency')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("Threshold Analysis:")
    print("Threshold Value | Vertices Kept | Percentage")
    print("-" * 45)
    for threshold in thresholds:
        vertices_to_remove = densities < threshold
        remaining = np.sum(~vertices_to_remove)
        percentage = (remaining / original_vertices) * 100
        print(f"{threshold:13.6f} | {remaining:11d} | {percentage:8.1f}%")

# Test different thresholds
test_thresholds(mesh, densities)

In [ ]:
def apply_threshold_and_visualize(mesh, densities, threshold):
    """Apply threshold and show before/after comparison"""
    
    # Get original data
    vertices = np.asarray(mesh.vertices)
    triangles = np.asarray(mesh.triangles)
    
    # Create mask for vertices to keep
    vertices_to_keep = densities >= threshold
    
    # Create mapping from old indices to new indices
    vertex_mapping = np.full(len(vertices), -1)
    vertex_mapping[vertices_to_keep] = np.arange(np.sum(vertices_to_keep))
    
    # Filter vertices
    new_vertices = vertices[vertices_to_keep]
    
    # Filter triangles - keep only triangles where all vertices are kept
    triangle_mask = np.all(vertices_to_keep[triangles], axis=1)
    new_triangles = triangles[triangle_mask]
    
    # Remap triangle indices
    new_triangles = vertex_mapping[new_triangles]
    
    # Create new mesh
    trimmed_mesh = o3d.geometry.TriangleMesh()
    trimmed_mesh.vertices = o3d.utility.Vector3dVector(new_vertices)
    trimmed_mesh.triangles = o3d.utility.Vector3iVector(new_triangles)
    
    # Copy colors if they exist
    if mesh.has_vertex_colors():
        vertex_colors = np.asarray(mesh.vertex_colors)
        new_vertex_colors = vertex_colors[vertices_to_keep]
        trimmed_mesh.vertex_colors = o3d.utility.Vector3dVector(new_vertex_colors)
    
    # Show statistics
    original_vertices = len(vertices)
    remaining_vertices = len(new_vertices)
    print(f"Original vertices: {original_vertices}")
    print(f"Remaining vertices: {remaining_vertices}")
    print(f"Removed: {original_vertices - remaining_vertices} ({100*(1-remaining_vertices/original_vertices):.1f}%)")
    
    # Visualize both
    print("Original mesh (press Q to close and see trimmed)")
    o3d.visualization.draw_geometries([mesh], window_name="Original Mesh")
    
    print("Trimmed mesh")
    o3d.visualization.draw_geometries([trimmed_mesh], window_name="Trimmed Mesh")
    
    return trimmed_mesh

# Example usage - choose threshold based on your analysis
selected_threshold = np.percentile(densities, 10)  # Adjust based on your plots
trimmed_mesh = apply_threshold_and_visualize(mesh, densities, selected_threshold)

In [ ]:
o3d.io.write_triangle_mesh("trimmed_leaf_mesh.ply", trimmed_mesh)

In [ ]:
pcd, mesh = mesh_leaf(leaf_np, method="poisson", backend="pymeshlab")

## Classical methods for surface reconstruction

### Apha shapes

In [ ]:
mesh_alpha = classical_methods.alpha_shape_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_alpha]) # -> Not bad but has holes
mesh_alpha.get_surface_area()

### Bezier

In [ ]:
mesh_bezier = classical_methods.bezier_surface_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_bezier]) # -> Has some issues, needs tuning
mesh_bezier.get_surface_area()

### B-spline

In [ ]:
mesh_bspline = classical_methods.bspline_surface_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_bspline]) # -> Looks better than bezier, but has some singularities
mesh_bspline.get_surface_area()

### Delaunay

In [ ]:
mesh_delaunay = classical_methods.delaunay_mesh_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_delaunay])
mesh_delaunay.get_surface_area()

### Poisson

In [ ]:
mesh_poisson = classical_methods.poisson_reconstruction(leaf_np, depth=9)
o3d.visualization.draw_geometries([mesh_poisson])
mesh_poisson.get_surface_area()

### Ball pivoting

In [ ]:
mesh_bpa = classical_methods.ball_pivoting_reconstruction(leaf_np, radius_multiplier=5.0)
o3d.visualization.draw_geometries([mesh_bpa])
mesh_bpa.get_surface_area()

Alternatively the area could be computed as below but it gives the same results

In [ ]:
vertices_bezier = np.asarray(mesh_bezier.vertices)
triangles_bezier = np.asarray(mesh_bezier.triangles)

area_bezier_manual = classical_methods.compute_mesh_area(vertices_bezier, triangles_bezier)
float(area_bezier_manual)